<a href="https://colab.research.google.com/github/dev-himakara/Lung-Tumor-Segmentation/blob/main/NN_Project_01_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1) Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 2) Install packages
# =========================
!pip install -q --upgrade idc-index
!pip install -q pandas==2.2.2

# =========================
# 3) Imports and paths
# =========================
import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/lung_nodule_project"
RAW_DIR = os.path.join(BASE_DIR, "lidc_subset_raw")
META_DIR = os.path.join(BASE_DIR, "metadata")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("META_DIR:", META_DIR)

# =========================
# 4) Initialize IDC client
# =========================
from idc_index import IDCClient
client = IDCClient()

print("Client initialized")
print([m for m in dir(client) if "patient" in m.lower() or "download" in m.lower()])

# =========================
# 5) Query patients in LIDC-IDRI
# IMPORTANT: use outputFormat="df"
# =========================
patients_df = client.get_patients("lidc_idri", outputFormat="df")

print(type(patients_df))
print("Columns:", patients_df.columns.tolist())
print("Total rows:", len(patients_df))
print(patients_df.head())

# =========================
# 6) Build unique patient list
# =========================
if "PatientID" not in patients_df.columns:
    raise ValueError(f"PatientID column not found. Available columns: {patients_df.columns.tolist()}")

unique_patients = patients_df["PatientID"].dropna().drop_duplicates().tolist()

print("Unique patients found:", len(unique_patients))
print("First 10 patients:", unique_patients[:10])

pd.Series(unique_patients).to_csv(
    os.path.join(META_DIR, "all_lidc_patients.csv"),
    index=False
)

# =========================
# 7) Test with 5 patients first.Then change it to number of datasets you want.
# =========================
TEST_PATIENTS = 180

selected_patients = (
    pd.Series(unique_patients)
    .sample(n=TEST_PATIENTS, random_state=42)
    .tolist()
)

print("Selected patients:", selected_patients)

pd.Series(selected_patients).to_csv(
    os.path.join(META_DIR, "selected_patients_test.csv"),
    index=False
)

# =========================
# 8) Download selected patients
# =========================
client.download_from_selection(
    patientId=selected_patients,
    downloadDir=RAW_DIR
)

print("Download complete")

# =========================
# 9) Inspect downloaded files
# =========================
found_any = False
for root, dirs, files in os.walk(RAW_DIR):
    if files:
        print("Folder:", root)
        print("Example files:", files[:10])
        found_any = True
        break

if not found_any:
    print("No files found yet. Check RAW_DIR manually.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RAW_DIR: /content/drive/MyDrive/lung_nodule_project/lidc_subset_raw
META_DIR: /content/drive/MyDrive/lung_nodule_project/metadata
Client initialized
['DOWNLOAD_HIERARCHY_DEFAULT', '_filter_by_patient_id', '_track_download_progress', '_validate_update_manifest_and_get_download_size', 'download_collection', 'download_dicom_instance', 'download_dicom_patients', 'download_dicom_series', 'download_dicom_studies', 'download_from_manifest', 'download_from_selection', 'get_patients']
<class 'pandas.core.frame.DataFrame'>
Columns: ['PatientID', 'PatientSex', 'PatientAge']
Total rows: 1010
        PatientID PatientSex PatientAge
0  LIDC-IDRI-0001       None       None
1  LIDC-IDRI-0002       None       None
2  LIDC-IDRI-0003       None       None
3  LIDC-IDRI-0004       None       None
4  LIDC-IDRI-0005       None       None
Unique patients found: 1010
First 10 patients: ['LIDC-IDRI-0001', 'LIDC-IDRI-0002', 'LIDC-IDRI-0003', 'LIDC-IDRI-0004', 'LIDC-IDRI-0005', 'LIDC-IDRI-0006', 'LIDC-IDRI-0007',

Download complete
Folder: /content/drive/MyDrive/lung_nodule_project/lidc_subset_raw/lidc_idri/LIDC-IDRI-0011/1.3.6.1.4.1.14519.5.2.1.6279.6001.292628672046109312619048073568/SEG_1.2.276.0.7230010.3.1.3.0.90085.1553284568.82945
Example files: ['093c75cb-334f-4944-9b0d-7fced1a23634.dcm']
